# NPE tuning stage 6

In [1]:
import numpy as np
from scipy import stats
import torch
from tqdm.auto import tqdm
import itertools
import torch
import torch.nn as nn
from torch.distributions import Uniform
import sbi
from sbi.utils.user_input_checks import MultipleIndependent
from sbi.neural_nets import posterior_nn
from sbi.neural_nets.embedding_nets import FCEmbedding
from sbi.inference import NPE_C
from sbi.diagnostics import run_sbc, check_sbc
import warnings
import sys
sys.path.append('../../pysimARG')
from discrete_uniform import DiscreteUniform
from LeaveLengthOut_NN import LeaveLengthOut_NN

torch_device = "cpu"

warnings.filterwarnings("ignore", category=UserWarning)

c:\Users\u2008181\likelihood-free\sbi_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load simulation data

Load genome data and clonal tree.

In [2]:
drop_col = range(16, 32)
theta_test = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/theta_sbc.csv', delimiter=",")
x_test = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/x_sbc.csv', delimiter=",")
x_test = np.delete(x_test, drop_col, axis=1)
print(theta_test.shape, x_test.shape)

nan_row_test = np.where(np.isnan(x_test) | np.isinf(x_test))[0]
print(nan_row_test)

theta_test = np.delete(theta_test, nan_row_test, axis=0)
theta_test = torch.tensor(theta_test, device=torch_device)
theta_test = theta_test.to(torch.float32)
theta_test_numpy = theta_test.cpu().numpy()

x_test = np.delete(x_test, nan_row_test, axis=0)
x_test = torch.tensor(x_test, device=torch_device)
x_test = x_test.to(torch.float32)
x_test_numpy = x_test.cpu().numpy()

print(theta_test.shape, x_test.shape)

(1000, 3) (1000, 30)
[865 899]
torch.Size([998, 3]) torch.Size([998, 30])


In [3]:
theta1 = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/theta1.csv', delimiter=",")
x1 = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/x1.csv', delimiter=",")
theta2 = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/theta2.csv', delimiter=",")
x2 = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/x2.csv', delimiter=",")

x = np.vstack([x1, x2])
x = np.delete(x, drop_col, axis=1)
theta = np.vstack([theta1, theta2])
print(theta.shape, x.shape)

nan_row = np.where(np.isnan(x) | np.isinf(x))[0]
print(nan_row)

theta = np.delete(theta, nan_row, axis=0)
theta = torch.tensor(theta[:10000, :], device=torch_device)
theta = theta.to(torch.float32)
theta_numpy = theta.cpu().numpy()

x = np.delete(x, nan_row, axis=0)
x = torch.tensor(x[:10000, :], device=torch_device)
x = x.to(torch.float32)
x_numpy = x.cpu().numpy()

print(theta.shape, x.shape)

(20000, 3) (20000, 30)
[  114   681   706  1448  2554  2818  7211  7282  7329  7392  8938  9827
  9973 10223 10788 12192 13567 14388 14653]
torch.Size([10000, 3]) torch.Size([10000, 30])


## Test functions

In [4]:
def SBC_KStest(ranks, num_posterior_samples, parameter_labels):
    num_dimensions = ranks.shape[1] 

    ks_results = []
    p_values = []
    for dim in range(num_dimensions):
        normalized_ranks = ranks[:, dim] / num_posterior_samples
        ks_stat, p_value = stats.kstest(normalized_ranks, 'uniform')
        ks_results.append(ks_stat)
        p_values.append(p_value)
    
    return ks_results, p_values

In [5]:
def mahalanobis_error(theta_est_post, theta_test_numpy):
    maha_errors = np.full((theta_est_post.shape[0]), np.nan)
    for i in range(theta_est_post.shape[0]):
        samples = theta_est_post[i]
        truth = theta_test_numpy[i]
        post_mean = np.mean(samples, axis=0)
        cov_matrix = np.cov(samples, rowvar=False)

        try:
            inv_cov = np.linalg.inv(cov_matrix)
        except np.linalg.LinAlgError:
            print(f"Warning: Singular covariance matrix at index {i}, returning NaN.")
            continue
        
        diff = post_mean - truth
        maha_dist_sq = np.dot(np.dot(diff, inv_cov), diff)
        maha_errors[i] = np.sqrt(maha_dist_sq)
    return maha_errors

## Tuning setting

In [6]:
seeds = [1, 2, 3, 4, 5]
num_posterior_samples = 1000
stage5_array = np.array([[2, 1, 128],
                         [2, 4, 256],
                         [2, 4, 48]])
print(stage5_array)

[[  2   1 128]
 [  2   4 256]
 [  2   4  48]]


In [7]:
stage5_indices = [0, 1, 2]
flow_transforms = [3, 5, 8]
hidden_features = [30, 50, 80, 120]
spline_bins = [5, 10, 20]
stage6_array = np.array(list(itertools.product(stage5_indices, flow_transforms, hidden_features, spline_bins)))
print(stage6_array)

[[  0   3  30   5]
 [  0   3  30  10]
 [  0   3  30  20]
 [  0   3  50   5]
 [  0   3  50  10]
 [  0   3  50  20]
 [  0   3  80   5]
 [  0   3  80  10]
 [  0   3  80  20]
 [  0   3 120   5]
 [  0   3 120  10]
 [  0   3 120  20]
 [  0   5  30   5]
 [  0   5  30  10]
 [  0   5  30  20]
 [  0   5  50   5]
 [  0   5  50  10]
 [  0   5  50  20]
 [  0   5  80   5]
 [  0   5  80  10]
 [  0   5  80  20]
 [  0   5 120   5]
 [  0   5 120  10]
 [  0   5 120  20]
 [  0   8  30   5]
 [  0   8  30  10]
 [  0   8  30  20]
 [  0   8  50   5]
 [  0   8  50  10]
 [  0   8  50  20]
 [  0   8  80   5]
 [  0   8  80  10]
 [  0   8  80  20]
 [  0   8 120   5]
 [  0   8 120  10]
 [  0   8 120  20]
 [  1   3  30   5]
 [  1   3  30  10]
 [  1   3  30  20]
 [  1   3  50   5]
 [  1   3  50  10]
 [  1   3  50  20]
 [  1   3  80   5]
 [  1   3  80  10]
 [  1   3  80  20]
 [  1   3 120   5]
 [  1   3 120  10]
 [  1   3 120  20]
 [  1   5  30   5]
 [  1   5  30  10]
 [  1   5  30  20]
 [  1   5  50   5]
 [  1   5  5

In [8]:
stage6_p_values = np.full((stage6_array.shape[0], 3), np.nan)
stage6_D_stats = np.full((stage6_array.shape[0], 3), np.nan)
stage6_maha_errors = np.full((stage6_array.shape[0]), np.nan)
stage6_nll = np.full((stage6_array.shape[0]), np.nan)

## Baseline NPE

In [9]:
prior_rho = Uniform(low=torch.tensor([0.0]), high=torch.tensor([0.1]))
prior_theta = Uniform(low=torch.tensor([0.0]), high=torch.tensor([0.1]))
prior_L = DiscreteUniform(low=torch.tensor([100.0]), high=torch.tensor([10000.0]))
prior = MultipleIndependent(
    dists=[prior_rho, prior_theta, prior_L],
    validate_args=False,
    device=torch_device
)

In [10]:
# for k in range(stage6_array.shape[0]):
#     # Stage 5 configuration
#     stage5_config = stage5_array[stage6_array[k, 0]]
#     num_outputs = stage5_config[0]
#     num_hidden_layers = stage5_config[1]
#     num_hiddens = stage5_config[2]

#     # Stage 6 configuration
#     num_transforms = stage6_array[k, 1]
#     num_features = stage6_array[k, 2]
#     num_bins = stage6_array[k, 3]

#     embedding_net = LeaveLengthOut_NN(
#         input_dim=30,
#         num_hiddens=num_hiddens,
#         num_hidden_layers=num_hidden_layers,
#         num_outputs=num_outputs)
#     neural_posterior = posterior_nn(
#         model="nsf",
#         embedding_net=embedding_net,
#         hidden_features=num_features,
#         num_transforms=num_transforms,
#         num_bins=num_bins
#     )
#     print(f"Running iteration {k}")
#     print(f"Stage 5 output dimension {num_outputs}, hidden layers {num_hidden_layers}, hidden units {num_hiddens}.")
#     print(f"Stage 6 transforms {num_transforms}, features {num_features}, bins {num_bins}.")
#     print("-" * 50)
    
#     seed = seeds[0]
#     torch.manual_seed(seed)
#     np.random.seed(seed)

#     inference_baseline = NPE_C(prior=prior, density_estimator=neural_posterior, device=torch_device)
#     density_estimator_baseline = inference_baseline.append_simulations(theta, x).train(
#         max_num_epochs=500
#     )
#     posterior_baseline = inference_baseline.build_posterior(density_estimator_baseline)

#     theta_est_post = np.full((theta_test.shape[0], num_posterior_samples, 3), np.nan)
#     for j in tqdm(range(theta_test.shape[0]), desc="Sampling posterior"):
#         theta_post = posterior_baseline.sample((num_posterior_samples,), x=x_test[j, :],
#                                             show_progress_bars=False, reject_outside_prior=False)
#         theta_est_post[j, :, :] = theta_post.detach().numpy()

#     parameter_labels = [r"for $\rho_s$", r"for $\theta_s$", r"for L"]
#     theta_test_expanded = theta_test.unsqueeze(1)
#     theta_est_post_tensor = torch.tensor(theta_est_post, device=torch_device)
#     theta_est_post_tensor = theta_est_post_tensor.to(torch.float32)
#     is_less_than_truth = theta_est_post_tensor < theta_test_expanded
#     ranks = torch.sum(is_less_than_truth, dim=1)

#     ks_results, p_values = SBC_KStest(ranks, num_posterior_samples, parameter_labels)
#     stage6_p_values[k, :] = p_values
#     stage6_D_stats[k, :] = ks_results

#     stage6_maha_errors[k] = np.mean(mahalanobis_error(theta_est_post, theta_test_numpy))

#     lp = density_estimator_baseline.log_prob(theta_test, x_test)
#     stage6_nll[k] = -lp.detach().cpu().mean().item()

In [11]:
# np.save('../../data/NPE_tuning/stage6_p_values.npy', stage6_p_values)
# np.save('../../data/NPE_tuning/stage6_D_stats.npy', stage6_D_stats)
# np.save('../../data/NPE_tuning/stage6_maha_errors.npy', stage6_maha_errors)
# np.save('../../data/NPE_tuning/stage6_nll.npy', stage6_nll)

## Load results and find the top three

In [12]:
stage6_p_values = np.load('../../data/NPE_tuning/stage6_p_values.npy')
stage6_D_stats = np.load('../../data/NPE_tuning/stage6_D_stats.npy')
stage6_maha_errors = np.load('../../data/NPE_tuning/stage6_maha_errors.npy')
stage6_nll = np.load('../../data/NPE_tuning/stage6_nll.npy')

In [13]:
stage6_nll.shape

(108,)

In [14]:
indices = np.argpartition(stage6_nll, 8)[:8]
print(indices)

[ 50  98  62  24  89  66 100  28]


In [15]:
stage6_array[indices]

array([[ 1,  5, 30, 20],
       [ 2,  8, 30, 20],
       [ 1,  8, 30, 20],
       [ 0,  8, 30,  5],
       [ 2,  5, 50, 20],
       [ 1,  8, 80,  5],
       [ 2,  8, 50, 10],
       [ 0,  8, 50, 10]])

In [16]:
stage6_p_values_final = np.full((len(seeds), 3, 8), np.nan)
stage6_D_stats_final = np.full((len(seeds), 3, 8), np.nan)
stage6_maha_errors_final = np.full((len(seeds), 8), np.nan)
stage6_nll_final = np.full((len(seeds), 8), np.nan)

In [19]:
for m in range(len(indices)):
    k = indices[m]
    # Stage 5 configuration
    stage5_config = stage5_array[stage6_array[k, 0]]
    num_outputs = stage5_config[0]
    num_hidden_layers = stage5_config[1]
    num_hiddens = stage5_config[2]

    # Stage 6 configuration
    num_transforms = stage6_array[k, 1]
    num_features = stage6_array[k, 2]
    num_bins = stage6_array[k, 3]

    embedding_net = LeaveLengthOut_NN(
        input_dim=30,
        num_hiddens=num_hiddens,
        num_hidden_layers=num_hidden_layers,
        num_outputs=num_outputs)
    neural_posterior = posterior_nn(
        model="nsf",
        embedding_net=embedding_net,
        hidden_features=num_features,
        num_transforms=num_transforms,
        num_bins=num_bins
    )
    print(f"Running index {k}")
    print(f"Stage 5 output dimension {num_outputs}, hidden layers {num_hidden_layers}, hidden units {num_hiddens}.")
    print(f"Stage 6 transforms {num_transforms}, features {num_features}, bins {num_bins}.")
    print("-" * 50)

    for i in range(len(seeds)):
        print(f"Running seed {seeds[i]}...")
        seed = seeds[0]
        torch.manual_seed(seed)
        np.random.seed(seed)

        inference_baseline = NPE_C(prior=prior, density_estimator=neural_posterior, device=torch_device)
        density_estimator_baseline = inference_baseline.append_simulations(theta, x).train(
            max_num_epochs=500
        )
        posterior_baseline = inference_baseline.build_posterior(density_estimator_baseline)

        theta_est_post = np.full((theta_test.shape[0], num_posterior_samples, 3), np.nan)
        for j in tqdm(range(theta_test.shape[0]), desc="Sampling posterior"):
            theta_post = posterior_baseline.sample((num_posterior_samples,), x=x_test[j, :],
                                                show_progress_bars=False, reject_outside_prior=False)
            theta_est_post[j, :, :] = theta_post.detach().numpy()

        parameter_labels = [r"for $\rho_s$", r"for $\theta_s$", r"for L"]
        theta_test_expanded = theta_test.unsqueeze(1)
        theta_est_post_tensor = torch.tensor(theta_est_post, device=torch_device)
        theta_est_post_tensor = theta_est_post_tensor.to(torch.float32)
        is_less_than_truth = theta_est_post_tensor < theta_test_expanded
        ranks = torch.sum(is_less_than_truth, dim=1)

        ks_results, p_values = SBC_KStest(ranks, num_posterior_samples, parameter_labels)
        stage6_p_values_final[i, :, m] = p_values
        stage6_D_stats_final[i, :, m] = ks_results

        stage6_maha_errors_final[i, m] = np.mean(mahalanobis_error(theta_est_post, theta_test_numpy))

        lp = density_estimator_baseline.log_prob(theta_test, x_test)
        stage6_nll_final[i, m] = -lp.detach().cpu().mean().item()

Running index 50
Stage 5 output dimension 2, hidden layers 4, hidden units 256.
Stage 6 transforms 5, features 30, bins 20.
--------------------------------------------------
Running seed 1...
 Neural network successfully converged after 137 epochs.

Sampling posterior: 100%|██████████| 998/998 [00:23<00:00, 42.34it/s]


Running seed 2...
 Neural network successfully converged after 149 epochs.

Sampling posterior: 100%|██████████| 998/998 [00:23<00:00, 41.81it/s]


Running seed 3...
 Neural network successfully converged after 130 epochs.

...

Sampling posterior: 100%|██████████| 998/998 [00:32<00:00, 30.89it/s]


Running seed 4...
 Neural network successfully converged after 107 epochs.

Sampling posterior: 100%|██████████| 998/998 [00:33<00:00, 29.72it/s]


Running seed 5...
 Neural network successfully converged after 105 epochs.

Sampling posterior: 100%|██████████| 998/998 [00:33<00:00, 29.97it/s]


In [20]:
np.save('../../data/NPE_tuning/stage6_p_values_final.npy', stage6_p_values_final)
np.save('../../data/NPE_tuning/stage6_D_stats_final.npy', stage6_D_stats_final)
np.save('../../data/NPE_tuning/stage6_maha_errors_final.npy', stage6_maha_errors_final)
np.save('../../data/NPE_tuning/stage6_nll_final.npy', stage6_nll_final)

In [21]:
print(np.mean(stage6_nll_final, axis=0))
print(np.median(stage6_nll_final, axis=0))

[-3.09280169 -3.06603374 -4.24498835 -4.49747419 -3.63606133 -4.0156179
 -4.4644423  -4.46117315]
[-2.37329936 -3.16152    -4.18939257 -4.52986002 -3.76242352 -4.12381601
 -4.61332035 -4.49555922]


In [22]:
print(np.mean(stage6_maha_errors_final, axis=0))
print(np.median(stage6_maha_errors_final, axis=0))

[1.25609604 1.23941924 1.15218291 0.98438175 1.01242052 0.89658208
 1.09984189 1.04149969]
[1.21763828 1.2496419  1.16676777 0.96013322 1.02154031 0.90153192
 1.10912777 1.01215912]


In [23]:
print(np.argsort(np.mean(stage6_nll_final, axis=0)))
print(np.argsort(np.median(stage6_nll_final, axis=0)))

[3 6 7 2 5 4 0 1]
[6 3 7 2 5 4 1 0]


In [24]:
indices[[3, 6, 7]]

array([ 24, 100,  28])

In [26]:
final_indices = [3, 6, 7]
print("Final Top 3 Configurations:")
print("-" * 50)
for i in final_indices:
    k = indices[i]
    # Stage 5 configuration
    stage5_config = stage5_array[stage6_array[k, 0]]
    num_outputs = stage5_config[0]
    num_hidden_layers = stage5_config[1]
    num_hiddens = stage5_config[2]

    # Stage 6 configuration
    num_transforms = stage6_array[k, 1]
    num_features = stage6_array[k, 2]
    num_bins = stage6_array[k, 3]
    print(f"Index {k}")
    print(f"Stage 5 output dimension {num_outputs}, hidden layers {num_hidden_layers}, hidden units {num_hiddens}.")
    print(f"Stage 6 transforms {num_transforms}, features {num_features}, bins {num_bins}.")
    print("-" * 50)

Final Top 3 Configurations:
--------------------------------------------------
Index 24
Stage 5 output dimension 2, hidden layers 1, hidden units 128.
Stage 6 transforms 8, features 30, bins 5.
--------------------------------------------------
Index 100
Stage 5 output dimension 2, hidden layers 4, hidden units 48.
Stage 6 transforms 8, features 50, bins 10.
--------------------------------------------------
Index 28
Stage 5 output dimension 2, hidden layers 1, hidden units 128.
Stage 6 transforms 8, features 50, bins 10.
--------------------------------------------------
